In [1]:
!pip install -q trl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 518.9/518.9 kB 9.2 MB/s eta 0:00:00ta 0:00:01


In [3]:
import os
import torch
from pathlib import Path
import json
from datasets import load_dataset
import pandas as pd
import numpy as np
from transformers import AutoTokenizer, AutoModelForCausalLM, TrainingArguments
from huggingface_hub import login
from kaggle_secrets import UserSecretsClient
from peft import LoraConfig, get_peft_model
from trl import SFTTrainer

2026-01-03 15:23:57.834131: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1767453838.049393      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1767453838.112492      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1767453838.637170      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767453838.637210      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1767453838.637213      55 computation_placer.cc:177] computation placer alr

In [15]:
import transformers
print(transformers.__version__)

4.57.1


## Preprocessing the dataset

In [4]:
MODEL_NAME="meta-llama/Llama-3.2-1B-Instruct"
LANGUAGE = "English"

DATA_PATH = f'/kaggle/input/chatbot-dataset/{LANGUAGE}_training_data.jsonl'

user_secrets = UserSecretsClient()
HF_TOKEN = user_secrets.get_secret("HF_TOKEN")
login(HF_TOKEN)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

dataset = load_dataset(
    "json",
    data_files=DATA_PATH,
    split="train"
)

tokenizer_config.json:   0%|          | 0.00/54.5k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/296 [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

### Functions Needed

In [5]:
# to format the text into template for Llama
def format_chat(example):
    text = tokenizer.apply_chat_template(
        example["messages"],
        tokenize=False,
        add_generation_prompt=False
    )
    return {"text": text}

# to check length of token
def compute_len(example):
    return {"len": len(tokenizer(example["text"])["input_ids"])}

In [6]:
# apply template
dataset = dataset.map(
    format_chat,
    remove_columns=dataset.column_names
)

lengths = dataset.map(
    compute_len
)

dataset= dataset.train_test_split(
    test_size=0.05,
    seed=42
)

Map:   0%|          | 0/3996 [00:00<?, ? examples/s]

Map:   0%|          | 0/3996 [00:00<?, ? examples/s]

In [20]:
print(dataset["train"][0]["text"])

<|begin_of_text|><|start_header_id|>system<|end_header_id|>

Cutting Knowledge Date: December 2023
Today Date: 03 Jan 2026

<|eot_id|><|start_header_id|>user<|end_header_id|>

I'm trying to be more healthy lately. I stopped to eat fast food last week.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

That's a great start! Fast food can be so convenient, but it's usually high in salt and sugar. How are you feeling since you changed your diet?<|eot_id|><|start_header_id|>user<|end_header_id|>

I feel more energy, but I'm still feeling bit tired in the mornings. I think I need more hours sleep.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Sleep is just as important as food. Most people need about seven or eight hours. Do you have a regular bedtime routine?<|eot_id|><|start_header_id|>user<|end_header_id|>

Not really. I usually watch my phone before sleep. Maybe that is why I cannot sleep fast.<|eot_id|><|start_header_id|>assistant<|end_header_id|>

Definitely! The blue l

### Checking token length to decide max len

In [7]:
lens = np.array(lengths['len'])

print(f"p50:   {np.percentile(lens, 50):.0f}")
print(f"p75:   {np.percentile(lens, 75):.0f}")
print(f"p90:   {np.percentile(lens, 90):.0f}")
print(f"p95:   {np.percentile(lens, 95):.0f}")
print(f"p99:   {np.percentile(lens, 99):.0f}")
print(f"max:   {lens.max()}")

p50:   245
p75:   319
p90:   398
p95:   451
p99:   556
max:   816


In [8]:
MAX_LEN = 896

## Prepare Model, Trainer, LoRA

In [9]:
base = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    dtype="auto",
    device_map="auto"
)

config.json:   0%|          | 0.00/877 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/2.47G [00:00<?, ?B/s]

generation_config.json:   0%|          | 0.00/189 [00:00<?, ?B/s]

In [10]:
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(base, lora_config)
model.print_trainable_parameters()

trainable params: 1,703,936 || all params: 1,237,518,336 || trainable%: 0.1377


In [25]:
training_args = TrainingArguments(
    output_dir=f"./kaggle/working/{LANGUAGE}/checkpoints",
    per_device_train_batch_size=2,
    gradient_accumulation_steps=4,
    learning_rate=2e-4,
    num_train_epochs=3,
    logging_steps=10,
    save_strategy="epoch",
    eval_strategy="epoch",
    fp16=True,
    report_to="none"
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    args=training_args,
)

trainer.train()

Adding EOS to train dataset:   0%|          | 0/3796 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/3796 [00:00<?, ? examples/s]

Truncating train dataset:   0%|          | 0/3796 [00:00<?, ? examples/s]

Adding EOS to eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Tokenizing eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

Truncating eval dataset:   0%|          | 0/200 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'eos_token_id': 128009, 'pad_token_id': 128009}.


Epoch,Training Loss,Validation Loss


KeyboardInterrupt: 

In [ ]:
final_dir = f"./kaggle/working/{LANGUAGE}/final"

trainer.model.save_pretrained(final_dir)
tokenizer.save_pretrained(final_dir)